In [1]:
!pip install timm librosa albumentations -q

In [ ]:
import os, gc, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import timm

warnings.filterwarnings("ignore")

In [ ]:
class CFG:
    BASE_DIR       = Path("/kaggle/input/competitions/birdclef-2026") 
    MODEL_DIR      = Path("/kaggle/input/datasets/mariia222/birdclef2026-models")
    OUTPUT_DIR     = Path("/kaggle/working")
    TAXON_CSV      = BASE_DIR / "taxonomy.csv"
    SAMPLE_SUB     = BASE_DIR / "sample_submission.csv"
    TEST_DIR       = BASE_DIR / "test_soundscapes"

    SR             = 32_000
    DURATION       = 5
    N_FFT          = 1024
    HOP_LENGTH     = 512
    N_MELS         = 128
    FMIN           = 50
    FMAX           = 14_000

    BATCH_SIZE     = 64
    NUM_WORKERS    = 2
    TTA            = 3      
    DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

    ENSEMBLE_W     = {"approach3": 1}

print(f"Device: {CFG.DEVICE}")

In [2]:



taxonomy   = pd.read_csv(CFG.TAXON_CSV)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB)

all_species = sorted(taxonomy["primary_label"].tolist())
NUM_CLASSES = len(all_species)
print(f"Classes: {NUM_CLASSES}  |  Sub rows: {len(sample_sub):,}")

def parse_row_id(row_id: str):
    """BC2026_Test_0001_S05_20250227_010002_20 → (filename, end_sec)"""
    parts    = row_id.rsplit("_", 1)
    end_time = int(parts[1])
    fname    = parts[0] + ".ogg"
    return fname, end_time

sub_meta = pd.DataFrame({"row_id": sample_sub["row_id"]})
sub_meta[["filename", "end_time"]] = sub_meta["row_id"].apply(
    lambda r: pd.Series(parse_row_id(r))
)
sub_meta["start_time"] = sub_meta["end_time"] - CFG.DURATION
sub_meta["filepath"]   = sub_meta["filename"].apply(
    lambda f: str(CFG.TEST_DIR / f)
)

print(f"Test soundscapes : {sub_meta['filename'].nunique()}")
print(f"Segments (rows)  : {len(sub_meta):,}")
print(sub_meta.head(3).to_string())

def load_clip(filepath, start, sr=CFG.SR, duration=CFG.DURATION):
    target = sr * duration
    
    if not os.path.exists(filepath):
        return np.zeros(target, dtype=np.float32)
        
    offset = max(0.0, float(start))
    try:
        y, _ = librosa.load(filepath, sr=sr, offset=offset, duration=duration)
    except Exception as e:
        
        print(f"Error loading {filepath}: {e}")
        return np.zeros(target, dtype=np.float32)
        
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
        
    return y[:target].astype(np.float32)
    
def compute_melspec(y):
    mel    = librosa.feature.melspectrogram(
        y=y, sr=CFG.SR,
        n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
        n_mels=CFG.N_MELS, fmin=CFG.FMIN, fmax=CFG.FMAX,
    )
    mel_db = librosa.power_to_db(mel, ref=1.0).astype(np.float32)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db


def compute_pcen(y):
    S      = np.abs(librosa.stft(y, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH)) ** 2
    mel_fb = librosa.filters.mel(
        sr=CFG.SR, n_fft=CFG.N_FFT, n_mels=CFG.N_MELS,
        fmin=CFG.FMIN, fmax=CFG.FMAX
    )
    mel    = mel_fb @ S
    pcen   = librosa.pcen(mel * (2**31), sr=CFG.SR,
                           hop_length=CFG.HOP_LENGTH).astype(np.float32)
    return (pcen - pcen.mean()) / (pcen.std() + 1e-6)

class TestDataset(Dataset):
    def __init__(self, df, use_pcen=False, noise_aug=False):
        self.df        = df.reset_index(drop=True)
        self.use_pcen  = use_pcen
        self.noise_aug = noise_aug

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y   = load_clip(row["filepath"], row["start_time"])

        if self.noise_aug:
            y = y + np.random.normal(0, 0.003, y.shape).astype(np.float32)

        feat = compute_pcen(y) if self.use_pcen else compute_melspec(y)
        feat = torch.from_numpy(np.stack([feat, feat, feat]))
        return feat, row["row_id"]
        
class BirdModel(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False, in_chans=3,
            num_classes=0, global_pool="avg"
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def load_model(model_name, weight_path, num_classes):
    model = BirdModel(model_name, num_classes).to(CFG.DEVICE)
    state = torch.load(weight_path, map_location=CFG.DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


@torch.no_grad()
def predict(model, df, use_pcen=False, tta=CFG.TTA):
    """Run inference with TTA, return (row_ids, preds) arrays."""
    all_preds, all_ids = [], []

    for tta_pass in range(tta):
        noise_aug = (tta_pass > 0)
        ds = TestDataset(df, use_pcen=use_pcen, noise_aug=noise_aug)
        dl = DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)

        pass_preds, pass_ids = [], []
        for mels, row_ids in dl:
            mels = mels.to(CFG.DEVICE)
            with autocast():
                out  = torch.sigmoid(model(mels)).cpu().numpy()
            pass_preds.append(out)
            if tta_pass == 0:
                pass_ids.extend(row_ids)

        all_preds.append(np.concatenate(pass_preds))
        if tta_pass == 0:
            all_ids = pass_ids

    avg_preds = np.mean(all_preds, axis=0)
    return all_ids, avg_preds


weighted_sum  = None
total_weight  = 0.0
final_row_ids = None

#w1_path = CFG.MODEL_DIR / "approach1_effb0_fold000.pth"
#if w1_path.exists():
#    print(f"\nRunning Approach 1: EfficientNet-B0 ...")
#    m1      = load_model("efficientnet_b0", w1_path, NUM_CLASSES)
#    ids, p1 = predict(m1, sub_meta, use_pcen=False)
#    w       = CFG.ENSEMBLE_W["approach1"]
#    weighted_sum  = p1 * w if weighted_sum is None else weighted_sum + p1 * w
#    total_weight += w
#    final_row_ids = ids
#    del m1; gc.collect(); torch.cuda.empty_cache()
#    print(f"  Approach 1 done. Shape: {p1.shape}")
#else:
#    print(f"[WARN] {w1_path} not found, skipping Approach 1")

#w2_path = CFG.MODEL_DIR / "approach2_effb2_mixup_fold0 (1).pth"
#if w2_path.exists():
#    print(f"\nRunning Approach 2: EfficientNet-B2 + MixUp ...")
#    m2      = load_model("efficientnet_b2", w2_path, NUM_CLASSES)
#    ids, p2 = predict(m2, sub_meta, use_pcen=False)
#    w       = CFG.ENSEMBLE_W["approach2"]
#    weighted_sum  = p2 * w if weighted_sum is None else weighted_sum + p2 * w
#    total_weight += w
#    if final_row_ids is None: final_row_ids = ids
#    del m2; gc.collect(); torch.cuda.empty_cache()
#    print(f"  Approach 2 done. Shape: {p2.shape}")
#else:
#    print(f"[WARN] {w2_path} not found, skipping Approach 2")

w3_path = CFG.MODEL_DIR / "approach3_pcen_fold0.pth"
if w3_path.exists():
    print(f"\nRunning Approach 3: PCEN + EfficientNet-B1 ...")
    m3      = load_model("efficientnet_b1", w3_path, NUM_CLASSES)
    ids, p3 = predict(m3, sub_meta, use_pcen=True)
    w       = CFG.ENSEMBLE_W["approach3"]
    weighted_sum  = p3 * w if weighted_sum is None else weighted_sum + p3 * w
    total_weight += w
    if final_row_ids is None: final_row_ids = ids
    del m3; gc.collect(); torch.cuda.empty_cache()
    print(f"  Approach 3 done. Shape: {p3.shape}")
else:
    print(f"[WARN] {w3_path} not found, skipping Approach 3")

if weighted_sum is None:
    raise RuntimeError("No model weights found! Upload trained weights to MODEL_DIR.")

final_preds = weighted_sum / (total_weight + 1e-9)

print(f"\nEnsemble weight sum : {total_weight:.2f}")
print(f"Final preds shape   : {final_preds.shape}")
print(f"Pred stats — min: {final_preds.min():.4f}  max: {final_preds.max():.4f}  mean: {final_preds.mean():.4f}")

species_cols = all_species  

submission          = pd.DataFrame(final_preds, columns=species_cols)
submission.insert(0, "row_id", final_row_ids)

submission = sample_sub[["row_id"]].merge(submission, on="row_id", how="left")
submission[species_cols] = submission[species_cols].fillna(0.0)

assert submission.shape == sample_sub.shape, \
    f"Shape mismatch! Got {submission.shape}, expected {sample_sub.shape}"

out_path = CFG.OUTPUT_DIR / "submission.csv"
submission.to_csv(out_path, index=False)


Device: cpu
Classes: 234  |  Sub rows: 3
Test soundscapes : 1
Segments (rows)  : 3
                                    row_id                                  filename  end_time  start_time                                                                                            filepath
0   BC2026_Test_0001_S05_20250227_010002_5  BC2026_Test_0001_S05_20250227_010002.ogg         5           0  /kaggle/input/competitions/birdclef-2026/test_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg
1  BC2026_Test_0001_S05_20250227_010002_10  BC2026_Test_0001_S05_20250227_010002.ogg        10           5  /kaggle/input/competitions/birdclef-2026/test_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg
2  BC2026_Test_0001_S05_20250227_010002_15  BC2026_Test_0001_S05_20250227_010002.ogg        15          10  /kaggle/input/competitions/birdclef-2026/test_soundscapes/BC2026_Test_0001_S05_20250227_010002.ogg

Running Approach 3: PCEN + EfficientNet-B1 ...
  Approach 3 done. Shape: (3, 234)

Ensem